In [2]:
# Core AI & Logic
!pip install -U langgraph langchain-google-genai

# UI & Web Features
!pip install streamlit streamlit-js-eval streamlit-notifications
!pip install pyngrok

# Location & Weather
!pip install requests geopy

# Image & Data Processing
!pip install pillow pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.8/173.8 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 3.4 MB/s eta 0:00:00
  Attempting uninstall: langgraph-prebuilt
    Found existing installation: langgraph-prebuilt 1.0.10
    Uninstalling langgraph-prebuilt-1.0.10:
      Successfully uninstalled langgraph-prebuilt-1.0.10
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.1.9
    Uninstalling langgraph-1.1.9:
      Successfully uninstalled langgraph-1.1.9
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 51.8 MB/s eta 0:00:00


In [4]:
!pip uninstall -y streamlit-notifications
!pip install streamlit-push-notifications

Found existing installation: streamlit-notifications 0.1
Uninstalling streamlit-notifications-0.1:
  Successfully uninstalled streamlit-notifications-0.1


In [10]:
%%writefile app.py
import streamlit as st
import os
import io
import json
import base64
import requests
from datetime import datetime
from typing import TypedDict, List, Optional
from PIL import Image
import time

# AI & Tools
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage
from geopy.geocoders import Nominatim
from streamlit_js_eval import get_geolocation

# --- 1. STATE & PERSISTENCE ---
class GardenerState(TypedDict):
    plant_id: str
    location: dict
    weather_data: dict
    health_logs: List[str]
    medication_plan: Optional[str]
    is_medication_due: bool
    current_image: str  # Base64 string
    analysis_output: str

def load_history(plant_id):
    if os.path.exists("garden_history.json"):
        with open("garden_history.json", "r") as f:
            data = json.load(f)
            return data.get(plant_id, [])
    return []

def save_history(plant_id, new_log):
    data = {}
    if os.path.exists("garden_history.json"):
        with open("garden_history.json", "r") as f:
            data = json.load(f)
    if plant_id not in data:
        data[plant_id] = []
    data[plant_id].append(new_log)
    with open("garden_history.json", "w") as f:
        json.dump(data, f)

# --- 2. THE THREE NODES ---

def environment_node(state: GardenerState):
    """Node 1: Assess hyperlocal environment."""
    lat, lon = state["location"].get("lat"), state["location"].get("lon")
    geolocator = Nominatim(user_agent="gardener_ai_lebanon")

    # Identify City/Town
    try:
        loc_data = geolocator.reverse(f"{lat}, {lon}")
        city = loc_data.raw.get('address', {}).get('city', 'Unknown City')
    except:
        city = "Location Hidden"

    # Fetch Weather
    api_key = os.environ.get("OPENWEATHER_API_KEY")
    url = f"https://api.openweathermap.org/data/2.5/weather?lat={lat}&lon={lon}&appid={api_key}&units=metric"

    try:
        res = requests.get(url).json()
        weather = {
            "temp": res['main']['temp'],
            "hum": res['main']['humidity'],
            "desc": res['weather'][0]['description']
        }
    except:
        weather = {"temp": "N/A", "hum": "N/A", "desc": "Offline"}

    state["weather_data"] = {**weather, "city": city}
    return state

def visual_expert_node(state: GardenerState):
    """Node 2: Identify plant and audit health relative to history."""
    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

    # Inject historical logs from JSON into the AI's prompt
    past_logs = load_history(state["plant_id"])
    context_str = "\n".join(past_logs[-3:]) # Fetch last 3 sessions
    for attempt in range(3):
        try:
            res = llm.invoke([msg])
            state["analysis_output"] = res.content
            break
        except Exception as e:
            if "503" in str(e) or "overloaded" in str(e).lower():
                st.warning(f"Gemini is busy, retrying in {2**attempt}s...")
                time.sleep(2**attempt)
            else:
                state["analysis_output"] = f"Error: {str(e)}"
                break

    prompt = f"""
    You are a professional Botanist.
    Location: {state['weather_data']['city']} ({state['weather_data']['temp']}°C).

    PREVIOUS HEALTH LOGS:
    {context_str if context_str else "No previous history found for this plant."}

    TASK:
    1. Identify the plant and assess current health.
    2. Analyze the image for pests, nutrient deficiency, or rot.
    3. COMPARE: Is the plant improving compared to the logs provided?
    """

    msg = HumanMessage(content=[
        {"type": "text", "text": prompt},
        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{state['current_image']}"}}
    ])

    res = llm.invoke([msg])
    state["analysis_output"] = res.content

    # Store a summary of this diagnosis in the JSON file
    summary = f"[{datetime.now().date()}] {res.content[:150]}..."
    save_history(state["plant_id"], summary)

    return state

def prescription_node(state: GardenerState):
    """Node 3: Determine if treatment is required."""
    output = state["analysis_output"].lower()
    # Simple logic: detect keywords that trigger a 'Medication Due' status
    keywords = ["fungus", "pest", "fertilizer", "repot", "disease", "insect"]
    needs_rx = any(k in output for k in keywords)

    state["is_medication_due"] = needs_rx
    state["medication_plan"] = "Apply recommended treatment from analysis" if needs_rx else "Keep current routine"
    return state

# --- 3. THE GRAPH ASSEMBLY (Manual Sequence) ---

def run_agent(initial_state: GardenerState):
    s1 = environment_node(initial_state)
    s2 = visual_expert_node(s1)
    s3 = prescription_node(s2)
    return s3

# --- 4. STREAMLIT UI ---

st.set_page_config(page_title="AI Gardener", layout="wide")
st.title("🌿 Agentic Gardener AI")
st.info("Using JSON Persistence (No SQLite Module Required)")

# Sidebar for History
plant_id = st.sidebar.text_input("Unique Plant Name", "My_Orchid")
st.sidebar.subheader("📜 Checkup History")
history = load_history(plant_id)
for log in reversed(history):
    st.sidebar.caption(log)

# Main UI
loc = get_geolocation()
img_file = st.camera_input("Check Plant Health")

if img_file and loc:
    b64_img = base64.b64encode(img_file.getvalue()).decode()

    initial_data = {
        "plant_id": plant_id,
        "location": {"lat": loc['coords']['latitude'], "lon": loc['coords']['longitude']},
        "current_image": b64_img,
        "health_logs": [],
        "weather_data": {}
    }

    with st.spinner("Processing through nodes..."):
        final_state = run_agent(initial_data)

        col1, col2 = st.columns(2)
        with col1:
            st.success("Analysis Complete")
            st.write(final_state["analysis_output"])

        with col2:
            st.metric("Temperature", f"{final_state['weather_data']['temp']}°C")
            st.metric("Humidity", f"{final_state['weather_data']['hum']}%")

            if final_state["is_medication_due"]:
                st.error(f"💊 Treatment Required: {final_state['medication_plan']}")
            else:
                st.balloons()
                st.write("✅ Plant is healthy!")

Overwriting app.py


In [11]:
from pyngrok import ngrok
from google.colab import userdata
import os

# Set environment variables from Colab Secrets
os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')
os.environ["OPENWEATHER_API_KEY"] = userdata.get('OPENWEATHER_API_KEY')
os.environ["NGROK_AUTH"] = userdata.get('NGROK_AUTH')

# 1. Start Tunnel
ngrok.kill()
ngrok.set_auth_token(os.environ["NGROK_AUTH"])
public_url = ngrok.connect(8501).public_url
print(f"🔗 APP LIVE AT: {public_url}")

# 2. Launch Streamlit
!streamlit run app.py --server.port 8501

🔗 APP LIVE AT: https://cocciferous-dioicous-gino.ngrok-free.dev


2026-05-11 10:14:15.811 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.81.203.114:8501



  Stopping...
  Stopping...
